# Module 2 - Variables and Data Types (Advanced)


The ELI5 notebook taught you how to use variables. This notebook explains **what happens underneath** when you create one. Understanding Python's internal model prevents an entire class of subtle bugs — the kind that appear when you copy a list, pass data between scripts, or process a large log file.

---

## Table of Contents

- [ ] 1. How Python Stores Variables in Memory
- [ ] 2. Dynamic Typing
- [ ] 3. Multiple Assignment
- [ ] 4. Mutable vs Immutable Types
- [ ] 5. f-string Format Specifiers
- [ ] 6. Type Conversion Edge Cases
- [ ] 7. Summary


---

## 1. How Python Stores Variables in Memory

**Analogy:** A variable name is not the box itself — it is a **post-it note** stuck to the box. The box (the object with the value) lives independently in memory. The post-it note (the variable name) points to it. Multiple post-it notes can point to the same box.

When you write:
```python
target_ip = "10.0.0.55"
```

Python does three things:
1. Creates a string object containing `"10.0.0.55"` somewhere in memory
2. Assigns it a unique address (its **identity**)
3. Binds the name `target_ip` to that address

```
target_ip  -------->  [Object: str "10.0.0.55"  |  id: 4498847088]
(the name)             (the actual data in memory)
```

The built-in `id()` function returns the memory address of any object. Two variables with the same `id` point to the **same object**.


In [ ]:
# id() returns the memory address of an object
target_ip  = "10.0.0.55"
backup_ip  = target_ip     # both names point to the same object

print("target_ip:", target_ip,  "| id:", id(target_ip))
print("backup_ip:", backup_ip,  "| id:", id(backup_ip))
print("Same object in memory:", id(target_ip) == id(backup_ip))


In [ ]:
# What happens when you reassign a variable?
# Python creates a NEW object and rebinds the name to it.
# The old object is not modified.

target_ip = "10.0.0.55"
print("Before reassignment - id:", id(target_ip))

target_ip = "10.0.0.99"   # new string object created, name rebound
print("After  reassignment - id:", id(target_ip))

# backup_ip still points to the old object
backup_ip = "10.0.0.55"
print("backup_ip unchanged:", backup_ip)


### Check Your Understanding: Object References

**Predict** — Without running the cell, decide: after line 3, will `host_a` and `host_b` have the same `id`? Why or why not?


In [ ]:
# Predict: same id or different id after each line?
host_a = "192.168.1.10"
host_b = host_a          # line 2
host_a = "192.168.1.20" # line 3

print("host_a:", host_a, "| id:", id(host_a))
print("host_b:", host_b, "| id:", id(host_b))
print("Same object:", id(host_a) == id(host_b))


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# After line 2: host_a and host_b both point to the same object ("192.168.1.10").
# After line 3: host_a is rebound to a NEW object ("192.168.1.20").
#               host_b still points to the original "192.168.1.10" object.
# Conclusion: strings are immutable. Reassigning a name never affects other names
#             that pointed to the old object.

host_a = "192.168.1.10"
host_b = host_a
host_a = "192.168.1.20"

print("host_a:", host_a)   # 192.168.1.20
print("host_b:", host_b)   # 192.168.1.10 -- unchanged
```

</details>

---

## 2. Dynamic Typing

Python is a **dynamically typed** language. This means the **value** carries the type, not the variable name. The same name can hold a string today and an integer tomorrow — Python does not care.

Compare this with a **statically typed** language like Java, where you must declare the type upfront and it is locked in:

```java
// Java: type is declared and cannot change
String hostname = "db-server";
hostname = 42;   // compile error!
```

```python
# Python: type follows the value, not the name
hostname = "db-server"   # str
hostname = 42            # now int — perfectly valid, but usually a bug
```

Dynamic typing makes Python flexible and fast to write, but it also means you need to be **more careful**: Python will not warn you at write time if you accidentally put the wrong type in a variable.


In [ ]:
# Python lets you reassign any type to any name
scan_result = "PASS"       # str
print(scan_result, type(scan_result))

scan_result = 0            # now int
print(scan_result, type(scan_result))

scan_result = True         # now bool
print(scan_result, type(scan_result))

scan_result = 0.0          # now float
print(scan_result, type(scan_result))


In [ ]:
# Why dynamic typing matters in practice:
# A log parser might receive a port as a string from a file, then convert it.
# If you do not convert it, arithmetic will fail silently or crash loudly.

port_from_log = "22"          # str: came in from a text file
print("Type from file:", type(port_from_log))

# Mistake: trying arithmetic on a string
# next_port = port_from_log + 1  # TypeError: can only concatenate str to str

# Correct: convert first
port_number = int(port_from_log)
next_port   = port_number + 1
print("Type after conversion:", type(port_number))
print("Next port:", next_port)


### Practice - Dynamic Typing

**Write** — The variable `raw_value` below holds the string `"9.8"`. Using `type()`, verify its type, convert it to a float named `cvss_score`, then verify the new type. Finally, classify it: if the score is above 9.0, set `severity` to `"CRITICAL"`, otherwise `"HIGH"`.

> **Note:** You do not know `if` statements yet — just hardcode the result based on the value. They will be covered in the Operators & Conditions module.


In [ ]:
# Your code here
raw_value = "9.8"


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
raw_value = "9.8"

print("Before conversion:", type(raw_value))   # str

cvss_score = float(raw_value)
print("After conversion: ", type(cvss_score))  # float

# 9.8 > 9.0, so severity is CRITICAL
severity = "CRITICAL"
print(f"CVSS {cvss_score} -> {severity}")
```

</details>

---

## 3. Multiple Assignment

Python lets you create multiple variables in a single line. This is useful for initialising a set of related values at once.

### Assigning the same value to multiple names

```python
a = b = c = 0
```


In [ ]:
# Reset multiple counters to zero in one line
failed_ssh = failed_http = failed_ftp = 0

print(failed_ssh, failed_http, failed_ftp)   # 0 0 0


### Tuple unpacking — assigning different values in one line

```python
name, value = "CVE-2024-1234", 9.8
```

The number of names on the left must match the number of values on the right.


In [ ]:
# Assign different values in one line — number of names must match number of values
cve_id, cvss_score, severity = "CVE-2024-1234", 9.8, "CRITICAL"

print(cve_id)      # CVE-2024-1234
print(cvss_score)  # 9.8
print(severity)    # CRITICAL


In [ ]:
# Augmented assignment: update a variable using its current value
# These are shortcuts for common update operations

failed_attempts = 2

failed_attempts += 1   # same as: failed_attempts = failed_attempts + 1
print("After += 1:", failed_attempts)  # 3

failed_attempts *= 2   # same as: failed_attempts = failed_attempts * 2
print("After *= 2:", failed_attempts)  # 6

# Also available: -=  /=  //=  %=  **=


### Practice - Multiple Assignment

**Write** — Using a single assignment line (tuple unpacking), create three variables:
- `source_ip` = `"203.0.113.5"`
- `destination_port` = `22`
- `protocol` = `"TCP"`

Then print them in this format: `203.0.113.5:22 via TCP`


In [ ]:
# Your code here


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
source_ip, destination_port, protocol = "203.0.113.5", 22, "TCP"

print(f"{source_ip}:{destination_port} via {protocol}")
```

</details>

---

## 4. Mutable vs Immutable Types

This is one of the most important distinctions in Python. It determines whether a value can be **changed in place** or whether changing it creates a new object entirely.

| Category | Types | Can be changed after creation? |
|----------|-------|---------------------------------|
| **Immutable** | `str`, `int`, `float`, `bool`, `tuple` | No — any "change" creates a new object |
| **Mutable** | `list`, `dict`, `set` | Yes — the object itself is modified |

### Immutable: strings, numbers, booleans

When you "change" an immutable value, Python creates a brand-new object and rebinds the variable name to it. The original object is left untouched.


In [ ]:
# Strings are immutable: "changing" creates a new object
hostname = "db-server-01"
print("Original id:", id(hostname))

hostname = "db-server-02"   # new string object created; name rebound
print("New id:     ", id(hostname))

# Different memory address: it really is a different object
# This is safe because no other variable is affected


In [ ]:
# You cannot modify a string in place
hostname = "db-server-01"

# This would crash: strings do not support item assignment
# hostname[0] = "X"   # TypeError: 'str' object does not support item assignment

# To get a modified version, create a NEW string using a method
updated_hostname = hostname.replace("01", "02")  # returns a new string
print("Original:", hostname)          # unchanged
print("Updated: ", updated_hostname)  # new object


### Mutable: lists (preview)

Lists are covered in depth in the Collections module. For now, note that **modifying a list changes the original object** — not just the variable pointing to it. This becomes critical when you copy a list or pass it around.

> You will see `list.append()` below. Full list operations are in the Collections module — this example is just to illustrate mutability.


In [ ]:
# Lists are mutable: modifying them changes the SAME object
open_ports = [22, 80, 443]
print("Before - id:", id(open_ports), "| value:", open_ports)

open_ports.append(8080)   # modifies the list in place
print("After  - id:", id(open_ports), "| value:", open_ports)

# Same memory address: the same object was modified, not replaced


In [ ]:
# The mutability trap: two names, one object
# When you write  b = a  with a mutable type,
# BOTH names point to the SAME list in memory.

blocklist_a = ["1.2.3.4", "5.6.7.8"]
blocklist_b = blocklist_a   # both names point to the same list

blocklist_a.append("9.10.11.12")   # modifies the shared object

print("blocklist_a:", blocklist_a)  # 3 items
print("blocklist_b:", blocklist_b)  # also 3 items! Same object.

# This surprises almost every beginner.
# The fix is to make an explicit copy — see the Collections module.


### Practice - Mutable vs Immutable

**Predict** — Before running the cell below, decide: what does `original_score` print on the last line, and why?


In [ ]:
# Predict the output of this cell
original_score = 9.8     # float — immutable
backup_score   = original_score
original_score = 7.5     # does this affect backup_score?

print("original_score:", original_score)
print("backup_score:  ", backup_score)


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# original_score was rebound to a new float object (7.5).
# backup_score still points to the original float object (9.8).
# Floats are immutable: reassigning original_score never touches backup_score.

original_score = 9.8
backup_score   = original_score
original_score = 7.5

print("original_score:", original_score)  # 7.5
print("backup_score:  ", backup_score)    # 9.8 (unchanged)
```

</details>

---

## 5. f-string Format Specifiers

f-strings can do more than just embed variables. Inside the `{}`, after a colon `:`, you can add a **format specifier** to control how the value is displayed.

| Specifier | Effect | Example |
|-----------|--------|---------|
| `:.2f` | Float, 2 decimal places | `f"{9.8:.2f}"` -> `"9.80"` |
| `:.1f` | Float, 1 decimal place | `f"{78.567:.1f}"` -> `"78.6"` |
| `:d` | Integer (explicit) | `f"{443:d}"` -> `"443"` |
| `:>10` | Right-align in 10 chars | `f"{22:>10}"` -> `"        22"` |
| `:<10` | Left-align in 10 chars | `f"{22:<10}"` -> `"22        "` |
| `:10` | Right-align (default for numbers) | `f"{22:10}"` -> `"        22"` |
| `:,` | Thousands separator | `f"{65535:,}"` -> `"65,535"` |


In [ ]:
# Controlling decimal places — useful for CVSS scores and percentages
cvss_score  = 9.8
cpu_usage   = 78.56789

print(f"CVSS score: {cvss_score:.1f}")    # 9.8
print(f"CVSS score: {cvss_score:.2f}")    # 9.80
print(f"CPU usage:  {cpu_usage:.1f}%")    # 78.6%


In [ ]:
# Column alignment — useful for building clean report tables
print(f"{'PORT':<8} {'SERVICE':<12} {'STATUS':<10}")
print("-" * 30)
print(f"{22:<8} {'SSH':<12} {'OPEN':<10}")
print(f"{80:<8} {'HTTP':<12} {'OPEN':<10}")
print(f"{443:<8} {'HTTPS':<12} {'OPEN':<10}")
print(f"{3389:<8} {'RDP':<12} {'FILTERED':<10}")


In [ ]:
# Thousands separator — useful for large counts
total_events   = 1234567
alerts_fired   = 4321

print(f"Total events processed: {total_events:,}")
print(f"Alerts fired:           {alerts_fired:,}")


### Practice - f-string Format Specifiers

**Write** — Build a scan summary using the variables below. The output must follow this exact format:

```
Scan Summary
============
Host:       10.0.0.55
Duration:   12.4 sec
Coverage:   1.52%
Events:     65,535
```

Use f-string format specifiers for the decimal and thousands separator.


In [ ]:
# Given — do not change
target_host   = "10.0.0.55"
duration_sec  = 12.4456
coverage_pct  = 1.51984
total_events  = 65535

# Your code here


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
target_host   = "10.0.0.55"
duration_sec  = 12.4456
coverage_pct  = 1.51984
total_events  = 65535

print("Scan Summary")
print("============")
print(f"{'Host:':<12}{target_host}")
print(f"{'Duration:':<12}{duration_sec:.1f} sec")
print(f"{'Coverage:':<12}{coverage_pct:.2f}%")
print(f"{'Events:':<12}{total_events:,}")
```

</details>

---

## 6. Type Conversion Edge Cases

Basic conversions like `int("443")` work smoothly. But real-world data is messy. Here are the edge cases you will encounter processing log files and config files.

### int() cannot handle decimals or non-numeric text


In [ ]:
# int() requires a clean integer string
print(int("443"))    # OK -> 443
print(int("-22"))    # OK -> -22 (negatives work)

# These would crash (shown as comments):
# int("22.5")   # ValueError: '22.5' has a decimal — use float() then int()
# int("22 ")    # Spaces? Nope — you need to strip() first
# int("PORT22") # ValueError: non-numeric characters

# Fix for decimal strings: convert to float first, then to int
score_str = "9.8"
score_int = int(float(score_str))   # 9.8 -> 9.0 -> 9  (truncates, does not round!)
print(f"int(float('{score_str}')) = {score_int}")  # 9 — notice truncation


In [ ]:
# strip() removes surrounding whitespace before conversion
# Common when reading from files or input with accidental spaces
dirty_port = "  443  "                  # whitespace from file read
clean_port = int(dirty_port.strip())    # strip() then int()

print(f"Cleaned port: {clean_port}")
print(f"Type: {type(clean_port)}")


In [ ]:
# bool() conversion: know the falsy values
# Falsy: 0, 0.0, "", None
# Truthy: everything else, including "False" (a non-empty string)

print(bool("False"))    # True! The string "False" is non-empty -> truthy
print(bool(""))         # False: empty string is falsy
print(bool(0))          # False: zero is falsy
print(bool("0"))        # True: string "0" is non-empty -> truthy

# This catches many beginners: "False" as a string is truthy


### Practice - Type Conversion Edge Cases

**Fix** — The script below reads port and score values from a log file (simulated as strings with whitespace). Fix each conversion so it produces the correct types and values without crashing.


In [ ]:
# Values as they arrive from a log file (messy strings)
raw_port  = "  8080  "     # has leading/trailing whitespace
raw_score = "  7.5  "      # same
raw_count = "  42  "       # same

# Fix these conversions — they will crash as-is
port_number = int(raw_port)          # Fix this line
cvss_score  = float(raw_score)       # Fix this line
event_count = int(raw_count)         # Fix this line

print(f"Port: {port_number} | Score: {cvss_score} | Count: {event_count}")


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
raw_port  = "  8080  "
raw_score = "  7.5  "
raw_count = "  42  "

# strip() removes whitespace before conversion
port_number = int(raw_port.strip())
cvss_score  = float(raw_score.strip())
event_count = int(raw_count.strip())

print(f"Port: {port_number} | Score: {cvss_score} | Count: {event_count}")
```

</details>

---

## 7. Summary

| Concept | Key rule |
|---------|----------|
| **Object references** | A variable name is a pointer to an object in memory. `id()` returns the memory address. |
| **Reassignment** | Rebinds the name to a new object. Never modifies the original object. |
| **Dynamic typing** | Types live on values, not on names. Python infers the type from the value you assign. |
| **Multiple assignment** | `a, b = 1, 2` unpacks values. `a = b = 0` assigns the same value to multiple names. |
| **Augmented assignment** | `x += 1` is a shortcut for `x = x + 1`. Also: `-=`, `*=`, `/=`. |
| **Immutable types** | `str`, `int`, `float`, `bool` — changing a value creates a new object. Safe to share names. |
| **Mutable types** | `list`, `dict`, `set` — modifications change the same object. Multiple names point to the same data. |
| **f-string specifiers** | `:.2f` (decimals), `:<10` (alignment), `:,` (thousands). Format without changing the variable. |
| **Conversion + strip()** | Always `.strip()` before converting strings from files or input. `int(float(...))` truncates. |

---

### Next Steps

You now understand how Python handles variables internally. These mechanics are foundational — you will rely on this knowledge throughout the rest of the curriculum.

**-> [02_Drills_Variables.ipynb](../01_variables/02_Drills_Variables.ipynb)** — 15 exercises covering ELI5 + Advanced material.
